# Fetching Data with an API
v.ekc

So far we've loaded data from files — but most real-world data lives behind APIs. Today you'll learn to request it programmatically using Python's `requests` library and the YouTube Data API.

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | Making a Request — YouTube Channel Endpoint |
| 3 | Parsing the Response |
| 4 | Enhancing with Video-Specific Info |
| 5 | 🔬 Open Exploration — Humboldt YouTube Data |
| Appendix | Quick Reference |

---
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd

---
## 2. Making a Request — YouTube Channel Endpoint

We'll use Google's YouTube Data API to pull information about Cal Poly Humboldt's channel.

### 📋 Board Reference

| Term | What it means |
|------|---------------|
| **API** | Application Programming Interface — a mechanism that lets two software components interact |
| **GET request** | Ask the server to send you data (most common API action) |
| **Endpoint** | The specific URL you're requesting data from |
| **Parameters** | Key-value pairs appended to the URL (`?key=abc&part=snippet`) |
| **Status code** | `200` = success · `400` = bad request · `403` = unauthorized · `404` = not found |
| **JSON** | The response format — structured like a Python dict |

> **Setup:** Go to [console.cloud.google.com](https://console.cloud.google.com/), create a project, enable the **YouTube Data API v3**, and copy your API Key below.

Read the YouTube API documentation [here](https://developers.google.com/youtube/v3/docs). We first want to get data about the YouTube channel — navigate to the Channel endpoint and read the documentation.

In [ ]:
# Paste your youtube API here
api_key = 

In [ ]:
# whenever we want to request data with an API
import requests

In [ ]:
# the url for making the request 
url = "https://www.googleapis.com/youtube/v3/channels?key="+api_key+"&part=snippet&forHandle=CalPolyHumboldt"

In [ ]:
# Request the data with a "GET" request
response = requests.get(url)
response

In [ ]:
# check out response
type(response)

In [ ]:
# learn more about response
help(response)

In [ ]:
# use the json() method to access the data
response.json()

In [ ]:
# Save the response data
payload = response.json()

In [ ]:
# inpsect the payload
payload.keys()

In [ ]:
# Inspect the data
payload['items'][0]['id']

In [ ]:
# Save the channel id
channel_id = payload['items'][0]['id']

In [ ]:
payload['items'][0]['snippet']

Now let's use the channel ID to fetch video data from the **search endpoint** — this time using a `params` dict instead of building the URL by hand.

In [ ]:
# Use this information to get video information (other notation)
search_url = 'https://www.googleapis.com/youtube/v3/search'
parameters = {'key':api_key,
         'part':'snippet',
         'channelId':channel_id,
         'order':'date',
          'maxResults':'50'}

search_response = requests.get(search_url, params = parameters)

In [ ]:
# check to make sure it was a successful request
search_response.status_code

In [ ]:
payload = search_response.json()

In [ ]:
payload.keys()

---
### Try It 1: Navigate the JSON Response 😊

1. How many items are in `payload['items']`? (Hint: `len()`)

2. What are the keys of `payload['items'][0]`?

3. Access the `title` of the first video in the payload. What nested path do you need?

4. What does `payload['nextPageToken']` contain? When might that be useful?

In [ ]:
# 1. How many items?


In [ ]:
# 2. Keys of the first item


In [ ]:
# 3. Title of the first video


In [ ]:
# 4. nextPageToken


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*`nextPageToken` lets you fetch the *next* page of results — useful when there are more results than `maxResults` allows in one call.*

```python
# 1.
len(payload['items'])

# 2.
payload['items'][0].keys()

# 3.
payload['items'][0]['snippet']['title']

# 4.
payload['nextPageToken']
```

</details>

---
## 3. Parsing/Preparing the Data

The raw JSON is nested — let's flatten it into a usable DataFrame.

### 📋 Board Reference

| Function | What it does |
|----------|--------------|
| `pd.DataFrame(payload['items'])` | Each item becomes a row — nested dicts stay as objects |
| `pd.json_normalize(payload['items'])` | Flattens nested dicts into separate columns (e.g., `snippet.title`) |
| `.drop(columns=[...])` | Remove unwanted columns |
| `.rename(columns={...})` | Rename specific columns |

In [ ]:
# Put the data in a pandas dataframe
payload_df = pd.DataFrame(payload['items'])
payload_df.head()

In [ ]:
payload_normalized_df = pd.json_normalize(payload['items'])
payload_normalized_df.head()

In [ ]:
payload_normalized_df.drop(columns = 'kind',inplace=True)

In [ ]:
# inspect the id data
payload_normalized_df.columns = ['_'.join(i.split('.')[-2:]) if 'snippet.thumbnails' in i 
                                 else i.split('.')[-1] for i in payload_normalized_df.columns]
payload_normalized_df.head()

In [ ]:
# Video title
clean_df = payload_normalized_df.copy()
clean_df.head()

---
## 4. Enhancing the Data with Video-Specific Info

The search endpoint gives us basic metadata — let's hit the **videos** endpoint to add view and like counts.

In [ ]:
# Test getting data for a specific video
video_id = "Wkbj2V8CQTw"
video_url = "https://www.googleapis.com/youtube/v3/videos"
video_params = {'key':api_key,
               'part':'statistics',
               'id':video_id}

In [ ]:
response_video_stats_test = requests.get(video_url,params = video_params)

In [ ]:
response_video_stats_test.json()

Get data for multiple videos:

In [ ]:
# get data for multiple videos
ids = ','.join(clean_df.videoId)
ids

In [ ]:
# Create parameters for more video requests
more_video_params = {'key':api_key,
               'part':'statistics',
               'id':ids}

In [ ]:
# Request the data
response_stats = requests.get(video_url, params = more_video_params).json()

In [ ]:
# Inspect the result
response_stats.keys()

In [ ]:
# Access the statistics
response_stats['items'][0]['statistics']

In [ ]:
# Add to the dataframe
clean_df['viewCount'] = [i['statistics']['viewCount'] for i in response_stats['items']]
clean_df.head()

In [ ]:
# Add to the dataframe
clean_df['likeCount'] = [i['statistics']['likeCount'] for i in response_stats['items']]
clean_df.head()

---
### Try It 2: Explore & Extend 😊

1. **Explore other endpoints:** Use the [YouTube API docs](https://developers.google.com/youtube/v3/docs) to get more information about Cal Poly Humboldt's channel or a specific video from a different endpoint or `part`.

2. **Choose your own channel:** With a partner, pick any YouTube channel and use `requests` to fetch basic video data (videoId, publishedAt, title).

3. **Engineer date columns:** From the `publishedAt` field, create two new columns — `date` (just the date) and `time` (just the time). *Hint: `publishedAt` is a string like `'2024-03-15T18:30:00Z'`.*

In [ ]:
# 1. Explore another endpoint or part


In [ ]:
# 2. Your channel — fetch video data


In [ ]:
# 3. Create date and time columns from publishedAt


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*`publishedAt` uses ISO 8601 format — the `T` separates date and time, and `Z` means UTC. You can split on `T` to get both parts.*

```python
# 3. Using string split
clean_df['date'] = clean_df['publishedAt'].str.split('T').str[0]
clean_df['time'] = clean_df['publishedAt'].str.split('T').str[1].str.replace('Z','')

# or with pd.to_datetime
clean_df['publishedAt_dt'] = pd.to_datetime(clean_df['publishedAt'])
clean_df['date'] = clean_df['publishedAt_dt'].dt.date
clean_df['time'] = clean_df['publishedAt_dt'].dt.time
```

</details>

---
## 5. 🔬 Open Exploration — Humboldt YouTube Data

No API key? No problem — use the pre-fetched dataset below. It contains real data pulled from Cal Poly Humboldt's YouTube channel.

| Column | Description |
|--------|-------------|
| `title` | Video title |
| `publishedAt` | Date/time the video was published |
| `viewCount` | Total views |
| `likeCount` | Total likes |
| `videoId` | YouTube video ID |

### ETL
1. Inspect dataset
2. Missing data
3. Data types
4. Summary statistics

In [ ]:
yt = pd.read_csv('youtube_humboldt.csv')
yt.head()

In [ ]:
yt.info()

In [ ]:
yt.describe()

**Prompt 1 — Distributions:**

Explore the distribution of `viewCount` or `likeCount`. What does the shape tell you about how views are distributed across Humboldt's videos?

In [ ]:
# Prompt 1


<details>
<summary>💡 One approach — click to peek</summary>
<br>

```python
import matplotlib.pyplot as plt
import seaborn as sns

yt['viewCount'] = pd.to_numeric(yt['viewCount'], errors='coerce')
sns.histplot(data=yt, x='viewCount', bins=30)
plt.title('Distribution of View Counts')
plt.show()
```

</details>

**Prompt 2 — Relationships:**

Is there a relationship between `likeCount` and `viewCount`? Make a scatter plot. Does the relationship look linear?

In [ ]:
# Prompt 2


<details>
<summary>💡 One approach — click to peek</summary>
<br>

```python
yt['likeCount'] = pd.to_numeric(yt['likeCount'], errors='coerce')
sns.scatterplot(data=yt, x='viewCount', y='likeCount')
plt.title('Views vs Likes')
plt.show()
```

</details>

**Prompt 3 — WiLdCaRd:**

Engineer a new column from the data (e.g., extract the year from `publishedAt`, create a `like_rate = likeCount / viewCount`). Use it in a plot. Requirements: at least 2 aesthetics or groupings.

*Your interpretation: (double-click to edit)*

In [ ]:
# Prompt 3 — your choice


<details>
<summary>💡 One approach — click to peek</summary>
<br>

*Many right answers — here's one using a derived `year` column.*

```python
yt['year'] = pd.to_datetime(yt['publishedAt']).dt.year
sns.boxplot(data=yt, x='year', y='viewCount')
plt.title('View Count by Year')
plt.show()
```

</details>

---
## Appendix — API Requests Quick Reference

### Minimal template
```python
import requests

response = requests.get('https://api.example.com/endpoint',
                        params={'key': api_key, 'param': 'value'})

print(response.status_code)  # 200 = success
data = response.json()       # parse JSON → Python dict
```

### Flatten nested JSON to DataFrame
```python
df = pd.DataFrame(data['items'])          # nested dicts stay as objects
df = pd.json_normalize(data['items'])     # flattens nested keys → columns
```

### Enrich with a second endpoint
```python
ids = ','.join(df['id_col'])              # combine ids for batch request
stats = requests.get(url, params={..., 'id': ids}).json()
df['col'] = [item['field'] for item in stats['items']]
```